# OSNet Training Walkthrough (Notebook)

This notebook is an interactive version of [src/training/train_osnet.py](src/training/train_osnet.py).

It helps you:
- understand each training step,
- run sanity training with your Market-1501 data,
- inspect loss/accuracy and checkpoint output.

## 1) Environment and Imports

This section sets project paths and imports the same modules as the script.

In [8]:
from pathlib import Path
import sys

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

# Resolve project root from notebook location: src/training/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from data.market1501_dataset import Market1501Dataset
from torchreid.reid.models import build_model

print(f"Project root: {PROJECT_ROOT}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Project root: C:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour
Torch version: 2.10.0+cu128
CUDA available: True


## 2) Configuration

Set hyperparameters here, equivalent to script arguments.

- `dataset_root`: Market-1501 root path
- `epochs`, `batch_size`: training length and memory usage
- `max_batches`: limit batches for quick sanity runs (`None` for full epoch)

In [9]:
dataset_root = r"C:\Users\redaj\Desktop\ME\THE-ONE\structued_dataset\Market-1501-v15.09.15"
epochs = 10
batch_size = 8
num_workers = 2
lr = 3e-4
weight_decay = 5e-4
max_batches = None
use_pretrained = True
save_path = PROJECT_ROOT / "checkpoints" / "osnet_market1501_notebook.pt"

print(f"dataset_root exists: {Path(dataset_root).exists()}")
print(f"save_path: {save_path}")

dataset_root exists: True
save_path: C:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour\checkpoints\osnet_market1501_notebook.pt


## 3) Build DataLoader

This replicates `build_dataloader(...)` from the script and returns:
- `train_loader`
- `num_classes` (number of identity labels in train split)

In [10]:
def build_dataloader(dataset_root: str, batch_size: int, num_workers: int):
    train_tfms = transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    train_ds = Market1501Dataset(root=dataset_root, split="train", transform=train_tfms)
    num_classes = len(train_ds.id_to_label)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
    )
    return train_loader, num_classes

train_loader, num_classes = build_dataloader(dataset_root, batch_size, num_workers)
print(f"Train classes: {num_classes}")
print(f"Batches/epoch: {len(train_loader)}")

Loaded train split: 12936 images, 751 identities
Train classes: 751
Batches/epoch: 1617


## 4) Build OSNet + Training Utilities

This creates OSNet for identity classification and defines one training epoch exactly like the script.

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = build_model(
    name="osnet_x1_0",
    num_classes=num_classes,
    loss="softmax",
    pretrained=use_pretrained,
    use_gpu=torch.cuda.is_available(),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


def train_one_epoch(model, loader, optimizer, criterion, device, max_batches=None):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for batch_idx, batch in enumerate(tqdm(loader, desc="Train", leave=False)):
        if max_batches is not None and batch_idx >= max_batches:
            break

        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    avg_loss = running_loss / max(1, running_total)
    avg_acc = running_correct / max(1, running_total)
    return avg_loss, avg_acc

Using device: cuda
Successfully loaded imagenet pretrained weights from "C:\Users\redaj/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


## 5) Run Training

Run this cell to train OSNet for the configured number of epochs.

Tip:
- For quick checks: keep `epochs=1` and `max_batches=20`
- For better learning: set `max_batches=None` and increase `epochs`

In [12]:
history = []
for epoch in range(1, epochs + 1):
    loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, device, max_batches)
    history.append({"epoch": epoch, "loss": loss, "acc": acc})
    print(f"Epoch {epoch}/{epochs} | loss={loss:.4f} | acc={acc:.4f}")

history

Epoch 1/10 | loss=3.9042 | acc=0.3271


Epoch 2/10 | loss=1.0644 | acc=0.7992


Epoch 3/10 | loss=0.4646 | acc=0.9168


Epoch 4/10 | loss=0.2878 | acc=0.9511


Epoch 5/10 | loss=0.2405 | acc=0.9609


...

Epoch 7/10 | loss=0.1976 | acc=0.9688


Epoch 8/10 | loss=0.1628 | acc=0.9761


Epoch 9/10 | loss=0.1599 | acc=0.9777


Epoch 10/10 | loss=0.1549 | acc=0.9791


[{'epoch': 1, 'loss': 3.9041932282362297, 'acc': 0.3270717377860235},
 {'epoch': 2, 'loss': 1.0644168645724075, 'acc': 0.7992424242424242},
 {'epoch': 3, 'loss': 0.4646372650125068, 'acc': 0.9168212739641312},
 {'epoch': 4, 'loss': 0.2877919215135318, 'acc': 0.9511440940012369},
 {'epoch': 5, 'loss': 0.24052298507533953, 'acc': 0.9608843537414966},
 {'epoch': 6, 'loss': 0.18515595234124174, 'acc': 0.9717068645640075},
 {'epoch': 7, 'loss': 0.19764459676851662, 'acc': 0.968769325912183},
 {'epoch': 8, 'loss': 0.16281809895970364, 'acc': 0.976113172541744},
 {'epoch': 9, 'loss': 0.1599320208955609, 'acc': 0.9777365491651205},
 {'epoch': 10, 'loss': 0.1548621833823211, 'acc': 0.9790507111935683}]

## 6) Save and Inspect Checkpoint

This writes the trained model state so you can reload it later from scripts or another notebook.

In [13]:
save_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_name": "osnet_x1_0",
        "num_classes": num_classes,
        "state_dict": model.state_dict(),
        "history": history,
    },
    save_path,
)
print(f"Checkpoint saved to: {save_path}")

ckpt = torch.load(save_path, map_location="cpu")
print(f"Saved model: {ckpt['model_name']}")
print(f"Saved classes: {ckpt['num_classes']}")
print(f"History entries: {len(ckpt.get('history', []))}")

Checkpoint saved to: C:\Users\redaj\Desktop\ME\THE-ONE\human_behaviour\checkpoints\osnet_market1501_notebook.pt
Saved model: osnet_x1_0
Saved classes: 751
History entries: 10


## 7) Optional: Quick Metrics View

This simple table makes epoch-by-epoch reading easier.

In [14]:
for row in history:
    print(f"epoch={row['epoch']:>2} | loss={row['loss']:.4f} | acc={row['acc']:.4f}")

epoch= 1 | loss=3.9042 | acc=0.3271
epoch= 2 | loss=1.0644 | acc=0.7992
epoch= 3 | loss=0.4646 | acc=0.9168
epoch= 4 | loss=0.2878 | acc=0.9511
epoch= 5 | loss=0.2405 | acc=0.9609
epoch= 6 | loss=0.1852 | acc=0.9717
epoch= 7 | loss=0.1976 | acc=0.9688
epoch= 8 | loss=0.1628 | acc=0.9761
epoch= 9 | loss=0.1599 | acc=0.9777
epoch=10 | loss=0.1549 | acc=0.9791


## 8) Re-ID Evaluation and Error Analysis

This section evaluates the trained model on **query vs gallery** and visualizes mistakes.

You will get:
- `mAP`, `Rank-1`, `Rank-5`, `Rank-10` (Market-1501 protocol)
- Top-1 retrieval accuracy (query-level)
- A confusion-style matrix (`true ID` vs `predicted top-1 ID`) for the most frequent IDs
- A bar chart of IDs with highest mistake rates

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SNS = True
except Exception:
    HAS_SNS = False

from data.market1501_dataset import Market1501Dataset, evaluate_reid


def extract_features_and_meta(model, dataloader, device):
    model.eval()
    feats, pids, cams, paths = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extract", leave=False):
            images = batch["image"].to(device)
            emb = model(images)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            feats.append(emb.cpu())
            pids.extend([int(x) for x in batch["person_id"]])
            cams.extend([int(x) for x in batch["camera_id"]])
            paths.extend(list(batch["path"]))

    features = torch.cat(feats, dim=0).numpy()
    return features, pids, cams, paths

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Build evaluation datasets/loaders
query_ds = Market1501Dataset(root=dataset_root, split="query")
gallery_ds = Market1501Dataset(root=dataset_root, split="gallery")

query_loader = DataLoader(query_ds, batch_size=64, shuffle=False, num_workers=num_workers)
gallery_loader = DataLoader(gallery_ds, batch_size=64, shuffle=False, num_workers=num_workers)

# Extract embeddings
q_feat, q_ids, q_cams, q_paths = extract_features_and_meta(model, query_loader, device)
g_feat, g_ids, g_cams, g_paths = extract_features_and_meta(model, gallery_loader, device)

# Global Re-ID metrics
metrics = evaluate_reid(
    query_features=q_feat,
    gallery_features=g_feat,
    query_ids=q_ids,
    gallery_ids=g_ids,
    query_cams=q_cams,
    gallery_cams=g_cams,
    metric="cosine",
)

print("Re-ID metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"- {k}: {v:.2f}")
    else:
        print(f"- {k}: {v}")

# Top-1 predicted ID per query (with Market-1501 exclusion rule)
q_mat = q_feat / (np.linalg.norm(q_feat, axis=1, keepdims=True) + 1e-12)
g_mat = g_feat / (np.linalg.norm(g_feat, axis=1, keepdims=True) + 1e-12)
sim = np.matmul(q_mat, g_mat.T)

pred_top1_ids = []
correct_top1 = []
for i in range(len(q_ids)):
    order = np.argsort(-sim[i])
    valid = [j for j in order if not (g_ids[j] == q_ids[i] and g_cams[j] == q_cams[i])]
    best_j = valid[0]
    pred_id = g_ids[best_j]
    pred_top1_ids.append(pred_id)
    correct_top1.append(int(pred_id == q_ids[i]))

top1 = 100.0 * np.mean(correct_top1)
print(f"Top-1 retrieval accuracy (query-level): {top1:.2f}%")

analysis_df = pd.DataFrame({
    "true_id": q_ids,
    "pred_id": pred_top1_ids,
    "correct": correct_top1,
})
analysis_df.head()

In [ ]:
# Confusion-style matrix on most frequent true IDs (readable subset)
TOP_IDS = 20
freq_ids = analysis_df["true_id"].value_counts().head(TOP_IDS).index.tolist()
subset = analysis_df[analysis_df["true_id"].isin(freq_ids)].copy()

# Keep only predictions inside the same frequent-ID set; others grouped as -9999 (OTHER)
subset["pred_group"] = subset["pred_id"].where(subset["pred_id"].isin(freq_ids), -9999)
labels = freq_ids + [-9999]
label_names = [str(x) for x in freq_ids] + ["OTHER"]

cm = pd.crosstab(
    pd.Categorical(subset["true_id"], categories=labels),
    pd.Categorical(subset["pred_group"], categories=labels),
    normalize="index",
).fillna(0.0)

plt.figure(figsize=(12, 9))
if HAS_SNS:
    sns.heatmap(cm, cmap="Blues", xticklabels=label_names, yticklabels=label_names)
else:
    plt.imshow(cm.values, cmap="Blues", aspect="auto")
    plt.xticks(ticks=np.arange(len(label_names)), labels=label_names, rotation=90)
    plt.yticks(ticks=np.arange(len(label_names)), labels=label_names)
    plt.colorbar()

plt.title("Top-1 ID Confusion (row-normalized) on frequent IDs")
plt.xlabel("Predicted ID")
plt.ylabel("True ID")
plt.tight_layout()
plt.show()

In [ ]:
# IDs with highest mistake rates
id_perf = analysis_df.groupby("true_id")["correct"].agg(["mean", "count"]).reset_index()
id_perf["error_rate"] = 1.0 - id_perf["mean"]
id_perf = id_perf[id_perf["count"] >= 2].sort_values("error_rate", ascending=False)

worst = id_perf.head(15)
plt.figure(figsize=(11, 5))
plt.bar(worst["true_id"].astype(str), worst["error_rate"])
plt.xticks(rotation=60)
plt.ylabel("Error rate")
plt.xlabel("True ID")
plt.title("IDs with highest Top-1 error rate")
plt.tight_layout()
plt.show()

worst.head(10)